# mem0 retrieval on LongMemEval_s, shard 1/3

Work kernel for [memory-recall-bench](https://github.com/Deepanjal-s/memory-recall-bench): mem0 2.2's own `search()` over verbatim messages (`add(infer=False)`, no LLM; fastembed bge-small, local Qdrant, spaCy; telemetry off). Output: rankings scored offline.

In [ ]:
!pip install -q "memory-recall-bench[mem0] @ git+https://github.com/Deepanjal-s/memory-recall-bench@v0.4.0"
!python -m spacy download -q en_core_web_sm

In [ ]:
import json, os, time, urllib.request
os.environ["MEM0_TELEMETRY"] = "False"
from pathlib import Path
from memory_recall_bench import load_longmemeval
from memory_recall_bench.integrations.mem0 import Mem0Retriever

SHARD, SHARDS = 0, 3
DATA = Path("longmemeval_s_cleaned.json")
if not DATA.exists():
    urllib.request.urlretrieve("https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json", DATA)
convs = sorted(load_longmemeval(DATA), key=lambda c: c.id)[SHARD::SHARDS]
print(len(convs), "conversations in this shard")
OUT = Path(f"/kaggle/working/mem0_rankings_shard{SHARD}.json")
done = json.loads(OUT.read_text()) if OUT.exists() else {}
r, t = Mem0Retriever(), time.time()
for c in convs:
    if c.id in done:
        continue
    done[c.id] = r.rank_all(c.messages, [c.question])[c.question]
    OUT.write_text(json.dumps(done))
    if len(done) % 10 == 0:
        print(len(done), f"done ({time.time() - t:.0f}s)", flush=True)
DATA.unlink()
print("finished", len(done))